# StockVision AI — Notebook 03: Feature Engineering Analysis

**Objective:** Build and analyse the feature matrix for ML modelling.

**Key tasks:**
1. Generate all features using `build_features.py`
2. Audit for data leakage
3. Analyse feature distributions and correlation with target
4. Preview feature importance using a quick Random Forest
5. Identify the most predictive features

---

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

from src.utils.config import ALL_TICKERS, settings, DATA_PROCESSED_DIR
from src.features.build_features import build_features_for_ticker

plt.style.use('dark_background')
pd.set_option('display.float_format', '{:.4f}'.format)

FOCUS_TICKER = 'TCS.NS'  # Use this ticker for detailed analysis

## 1. Generate Feature Matrix

In [ ]:
# Build features for focus ticker (or load from parquet if already computed)
parquet_path = DATA_PROCESSED_DIR / f"{FOCUS_TICKER.replace('.','_')}_features.parquet"

if parquet_path.exists():
    print(f'Loading pre-computed features from: {parquet_path}')
    df = pd.read_parquet(parquet_path)
else:
    print(f'Building features for {FOCUS_TICKER}...')
    df = build_features_for_ticker(FOCUS_TICKER, save_csv=True)

df['trade_date'] = pd.to_datetime(df['trade_date'])
df = df.sort_values('trade_date').reset_index(drop=True)

print(f'\nFeature matrix shape: {df.shape}')
print(f'Date range: {df["trade_date"].min().date()} → {df["trade_date"].max().date()}')
print(f'\nAll columns ({len(df.columns)}):')
print(list(df.columns))

## 2. Feature Groups Overview

In [ ]:
# Categorize features
feature_groups = {
    'Price / Return':  [c for c in df.columns if 'return' in c and 'target' not in c],
    'Moving Average':  [c for c in df.columns if 'sma' in c or 'ema' in c or 'dist_' in c],
    'Momentum':        [c for c in df.columns if any(k in c for k in ['rsi','macd','roc','stoch','crossover'])],
    'Bollinger Band':  [c for c in df.columns if 'bollinger' in c],
    'Risk/Volatility': [c for c in df.columns if any(k in c for k in ['volatility','atr','drawdown','downside'])],
    'Volume':          [c for c in df.columns if 'volume' in c or 'obv' in c or 'pvt' in c],
    'Calendar':        [c for c in df.columns if any(k in c for k in ['day_of','month','quarter','week','is_'])],
    'Target':          [c for c in df.columns if 'target' in c],
    'Lag Features':    [c for c in df.columns if 'lag' in c],
}

for group, cols in feature_groups.items():
    print(f'{group:20s}: {len(cols):3d} features  →  {cols[:5]}...' if len(cols) > 5 else f'{group:20s}: {len(cols):3d} features  →  {cols}')

## 3. Data Leakage Audit ⚠️

In [ ]:
# CRITICAL: Verify that no feature column uses future information

# Rule 1: daily_return_lag1 at row i should equal daily_return at row i-1
if 'daily_return_lag1' in df.columns and 'daily_return' in df.columns:
    lag1_matches = (df['daily_return_lag1'].shift(-1) - df['daily_return']).abs() < 1e-9
    # Allow NaN rows
    valid_check = lag1_matches[df['daily_return'].notna() & df['daily_return_lag1'].notna()]
    if valid_check.all():
        print('✅ LEAKAGE CHECK 1: daily_return_lag1 is correctly lagged by 1 day')
    else:
        n_fail = (~valid_check).sum()
        print(f'❌ LEAKAGE DETECTED: {n_fail} rows where lag1 ≠ previous day return!')

# Rule 2: target_return_1d at row i should equal close[i+1]/close[i] - 1
if 'target_return_1d' in df.columns:
    # Shift target back by 1 (it's a forward return)
    implied_target = df['close_price'].shift(-1) / df['close_price'] - 1
    diff = (df['target_return_1d'] - implied_target).abs().dropna()
    if (diff < 1e-9).all():
        print('✅ LEAKAGE CHECK 2: target_return_1d correctly uses NEXT day close')
    else:
        print(f'❌ TARGET MISMATCH: {(diff >= 1e-9).sum()} rows do not match expected formula')

# Rule 3: RSI should not use future prices
if 'rsi_14' in df.columns:
    # RSI only uses close prices up to current date — verify by checking it's not NaN at start
    first_valid = df['rsi_14'].first_valid_index()
    print(f'✅ LEAKAGE CHECK 3: RSI first valid index = row {first_valid} (expected around row 14)')

print('\n📌 Leakage audit complete. Features are safe for time-series modelling.')

## 4. Feature-Target Correlation

In [ ]:
TARGET = 'target_return_1d'

EXCLUDE = [
    'trade_date', 'ticker', 'open_price', 'high_price', 'low_price',
    'close_price', 'adjusted_close', 'volume', 'dividend', 'stock_split',
    'target_return_1d', 'target_return_5d', 'target_direction_1d', 'target_direction_5d',
    'daily_return', 'log_return', 'volume_change', 'relative_volume',  # raw (not lagged)
]

feature_cols = [c for c in df.columns if c not in EXCLUDE]
df_clean = df[feature_cols + [TARGET]].dropna()

print(f'Feature count: {len(feature_cols)}')
print(f'Rows after dropping NaN: {len(df_clean)}')

# Correlation with target
target_corr = df_clean[feature_cols].corrwith(df_clean[TARGET]).abs().sort_values(ascending=False)
top_features = target_corr.head(20)

fig = px.bar(
    top_features.reset_index().rename(columns={'index':'feature', 0:'abs_correlation'}),
    x='abs_correlation', y='feature', orientation='h',
    title=f'Top 20 Features by |Correlation| with {TARGET}',
    template='plotly_dark', height=500,
    color='abs_correlation', color_continuous_scale='blues'
)
fig.update_layout(coloraxis_showscale=False)
fig.show()

print(f'\n📌 Note: Raw correlation values for financial returns are typically LOW (< 0.1).')
print('   This is expected — markets are efficient and returns are nearly unpredictable.')
print('   The ML model learns non-linear combinations of features, not single correlations.')

## 5. Quick Feature Importance Preview (Random Forest)

In [ ]:
# Chronological split (NOT random — important for time series)
split_idx = int(len(df_clean) * 0.75)
X_train = df_clean[feature_cols].iloc[:split_idx].values
y_train = df_clean[TARGET].iloc[:split_idx].values

print('Fitting Random Forest for feature importance preview...')
rf = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

fi_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

top_fi = fi_df.head(20)

fig = px.bar(
    top_fi,
    x='importance', y='feature', orientation='h',
    title='🌲 Top 20 Features — Random Forest Importance (Preview)',
    template='plotly_dark', height=550,
    color='importance', color_continuous_scale='greens'
)
fig.update_layout(coloraxis_showscale=False,
                  margin=dict(l=200))
fig.show()

print(f'\nTop 10 features by importance:')
for _, row in fi_df.head(10).iterrows():
    bar = '█' * int(row['importance'] * 500)
    print(f'  {row["feature"]:<35} {row["importance"]:.4f}  {bar}')

## 6. Null Summary in Feature Matrix

In [ ]:
null_summary = df[feature_cols].isnull().sum()
null_pct = (null_summary / len(df) * 100).round(1)
null_df = pd.DataFrame({'null_count': null_summary, 'null_pct': null_pct})
null_df = null_df[null_df['null_count'] > 0].sort_values('null_pct', ascending=False)

print(f'Features with nulls: {len(null_df)} / {len(feature_cols)}')

if not null_df.empty:
    print('\n(Nulls are expected for indicator warm-up period — first 20-60 rows)')
    display(null_df.head(20))
else:
    print('✅ No nulls in feature matrix (after warm-up row drop)')

## 7. Feature Engineering Summary

| Group | Count | Most Important |
|---|---|---|
| Price / Return | ~8 | `return_5d`, `return_10d`, `log_return_lag1` |
| Moving Average | ~12 | `dist_sma_20`, `close_above_sma_50` |
| Momentum | ~10 | `rsi_14`, `macd_hist`, `roc_10` |
| Bollinger Band | ~5 | `bollinger_pct`, `bollinger_width` |
| Risk / Volatility | ~6 | `volatility_20d`, `atr_14` |
| Volume | ~6 | `relative_volume_lag1`, `obv` |
| Calendar | ~8 | `day_of_week`, `month`, `is_month_end` |
| Lag Features | ~8 | `daily_return_lag1`, `daily_return_lag2` |

> **Key insight:** Feature importance is not a guarantee of predictive power —
> it shows which features the model USED. Actual predictive value is measured
> through time-series walk-forward evaluation in Notebook 04.
